In [1]:
import torch
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.llms import Ollama

# Reverting to your original, correct imports!
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_core.prompts import ChatPromptTemplate

In [2]:
import time
import concurrent.futures
from pathlib import Path
from io import BytesIO

from pdf2image import convert_from_path
import pytesseract
from pypdf import PdfReader, PdfWriter

# --- 1. Modified Worker Function ---
def process_single_page(image):
    """
    Returns RAW BYTES instead of a complex PDF object.
    Raw bytes can be safely passed between CPU cores without crashing.
    """
    return pytesseract.image_to_pdf_or_hocr(image, extension="pdf")

# --- 2. Main Execution ---
if __name__ == '__main__':
    input_pdf = Path("/Users/saketh/PennyStockLLM/Penny_stock.pdf")
    output_pdf = Path("Penny_stock_ocr.pdf")
    writer = PdfWriter()

    start_all = time.perf_counter()
    print("Extracting images from PDF (this might take a moment)...")
    
    # Extract images using threads
    images = convert_from_path(str(input_pdf), dpi=300, thread_count=12)
    print(f"Extracted {len(images)} images.")

    print(f"Starting Multi-core OCR processing on {len(images)} pages...")
    
    # Use threads because PIL image objects cannot be reliably pickled across processes
    with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
        ocr_byte_results = list(executor.map(process_single_page, images))

    print("Stitching PDF back together in the main process...")
    # Now we parse those bytes into PDF pages safely in the main thread
    for idx, pdf_bytes in enumerate(ocr_byte_results, start=1):
        reader = PdfReader(BytesIO(pdf_bytes))
        writer.add_page(reader.pages[0])
        
        # Print a progress update every 10 pages
        if idx % 10 == 0 or idx == len(ocr_byte_results):
            print(f"Stitched {idx}/{len(ocr_byte_results)} pages")

    print("Saving file to disk...")
    with output_pdf.open("wb") as f:
        writer.write(f)

    print("Saved searchable PDF:", output_pdf)

    total_elapsed = time.perf_counter() - start_all
    print(f"Total time: {total_elapsed:.2f} seconds")

Extracting images from PDF (this might take a moment)...
Extracted 205 images.
Starting Multi-core OCR processing on 205 pages...
Stitching PDF back together in the main process...
Stitched 10/205 pages
Stitched 20/205 pages
Stitched 30/205 pages
Stitched 40/205 pages
Stitched 50/205 pages
Stitched 60/205 pages
Stitched 70/205 pages
Stitched 80/205 pages
Stitched 90/205 pages
Stitched 100/205 pages
Stitched 110/205 pages
Stitched 120/205 pages
Stitched 130/205 pages
Stitched 140/205 pages
Stitched 150/205 pages
Stitched 160/205 pages
Stitched 170/205 pages
Stitched 180/205 pages
Stitched 190/205 pages
Stitched 200/205 pages
Stitched 205/205 pages
Saving file to disk...
Saved searchable PDF: Penny_stock_ocr.pdf
Total time: 66.24 seconds


In [3]:
#Embeddings process

PDF_PATH = "Penny_stock_ocr.pdf"
DB_DIR = "./penny_stock_vector_db"

print("Loading PDF...")
loader = PyPDFLoader(PDF_PATH)
# Use fast mode by avoiding unnecessary metadata parsing
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2000,
    chunk_overlap = 400,
    add_start_index = True,
)

chunks = text_splitter.split_documents(docs)
print(f"Generated {len(chunks)} semantic chunks.")

# 3. GPU-ACCELERATED EMBEDDINGS
print("Initializing GPU-accelerated embeddings...")

device = "mps" if torch.backends.mps.is_available() else "cpu"

model_kwargs = {'device': device}
encode_kwargs = {'normalize_embeddings': True}

embeddings = HuggingFaceEmbeddings(
    model_name = "BAAI/bge-large-en-v1.5",
    model_kwargs = model_kwargs,
    encode_kwargs = encode_kwargs
)

# 4. BATCHED VECTOR STORAGE
print("Storing in ChromaDB (using GPU)...")
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_DIR,
    collection_name="langchain_large",
    collection_metadata={"hnsw:space": "cosine"}
)

print(f"Optimization Complete! Vector DB saved at: {DB_DIR}")



Loading PDF...
Generated 206 semantic chunks.
Initializing GPU-accelerated embeddings...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Storing in ChromaDB (using GPU)...
Optimization Complete! Vector DB saved at: ./penny_stock_vector_db


In [ ]:
#Establishment of connection to embeddings & database
print("Connecting to local Vector DB...")
device = "mps" if torch.backends.mps.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name = "BAAI/bge-large-en-v1.5",
    model_kwargs={'device':device},
    encode_kwargs={'normalize_embeddings': True}
)

vector_db = Chroma(
    persist_directory="./penny_stock_vector_db",
    embedding_function=embeddings,
    collection_name="langchain_large"
)

# 2. SETUP RETRIEVER
retriever = vector_db.as_retriever(search_type="mmr",search_kwargs={"k": 8, "fetch_k": 30})

# 3. INITIALIZE THE LLM
print("Waking up Qwen 3.5 (35B)...")
llm = Ollama(model="qwen3.5:35b")

# 4. CREATE THE SYSTEM PROMPT

system_prompt = (
    "You are a sharp, analytical Financial SDE. Use the provided context to answer the user's question. "
    "If the context does not contain the answer, explicitly state: 'The provided document does not contain this information.' "
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

#5. ASSEMBLE THE RAG CHAIN
print("Assembling RAG Chain...")
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

#6. EXECUTE 
query = "If I have Rs 100, calculate how many average penny stocks I could buy based on the context, and evaluate if this is a safe investment compared to large-cap stocks."
response = rag_chain.invoke({"input": query})

print("\n--- AI ANALYSIS ---")
print(response["answer"])



Connecting to local Vector DB...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Waking up Qwen 3.5 (35B)...
Assembling RAG Chain...


/var/folders/9l/yjhwmfrs4m379sjy1069t9lm0000gn/T/ipykernel_71637/2854885463.py:22: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="qwen3.5:35b")



--- AI ANALYSIS ---
Based on the provided context, I can evaluate the safety aspect, but I cannot provide a precise calculation for the number of shares due to missing data.

**1. Calculation of Shares (Rs 100 Investment)**
The provided document **does not contain the specific information** required to calculate the exact number of average penny stocks purchasable for Rs 100.
*   **Definition:** The text defines penny stocks generally as those "priced lower than Rs 10."
*   **Price Variability:** While it mentions specific examples like Chemcrux Enterprises (adjusted to Rs 5-6) or historical ranges (Rs 10-12, Rs 15-20 adjusted), it does not provide an "average price" for penny stocks across the market.
*   **Inference:** Theoretically, since penny stocks are priced below Rs 10, Rs 100 could purchase more than 10 shares (potentially up to 20 or more if prices are significantly lower), but an exact calculation is not possible from the text alone.

**2. Safety Evaluation: Penny Stocks vs